In [21]:
import numpy as np
import scipy.io as sio


#load dataset

data = sio.loadmat("./data/DREAMER.mat")
mat = sio.loadmat(
    "data/DREAMER.mat",
    squeeze_me = True,
    struct_as_record=False
)


dreamer = mat["DREAMER"]

In [22]:
#dataset overview
print("Number of subjects:", dreamer.noOfSubjects)
print("Number of videos:", dreamer.noOfVideoSequences)
print("EEG sampling rate:", dreamer.EEG_SamplingRate)
print("ECG sampling rate:", dreamer.ECG_SamplingRate)

print("\nEEG electrodes:")
print(dreamer.EEG_Electrodes)

Number of subjects: 23
Number of videos: 18
EEG sampling rate: 128
ECG sampling rate: 256

EEG electrodes:
['AF3' 'F7' 'F3' 'FC5' 'T7' 'P7' 'O1' 'O2' 'P8' 'T8' 'FC6' 'F4' 'F8' 'AF4']


In [23]:
#extract EEG 
all_eeg = []

for s in range(dreamer.noOfSubjects):
    subject = dreamer.Data[s]

    #EEG structure
    eeg_struct = subject.EEG

    stimuli = eeg_struct.stimuli

    subject_eeg = []

    for video in range(dreamer.noOfVideoSequences):
        eeg = np.asarray(stimuli[video])
        subject_eeg.append(eeg)

    all_eeg.append(subject_eeg)


In [24]:
# Convert to NumPy object array
all_eeg = np.array(all_eeg , dtype=object)
print("Array shape:", all_eeg.shape)
print("Array dtype:", all_eeg.dtype)

Array shape: (23, 18)
Array dtype: object


In [25]:
#checking first record
first = all_eeg[0,0]
print("\nFirst recording:")
print("Subject: 1")
print("Video: 1")
print("Shape:", first.shape)

print("\nFirst 5 samples:")
print(first[:5])


First recording:
Subject: 1
Video: 1
Shape: (25472, 14)

First 5 samples:
[[4388.20512821 4102.56410256 4219.48717949 4465.12820513 4370.76923077
  4399.48717949 4443.07692308 4023.07692308 4365.12820513 4310.25641026
  3953.84615385 4454.35897436 4326.15384615 4165.12820513]
 [4375.8974359  4093.84615385 4252.82051282 4522.56410256 4435.8974359
  4411.79487179 4488.71794872 4108.71794872 4399.48717949 4384.61538462
  4007.69230769 4466.66666667 4372.82051282 4247.17948718]
 [4378.46153846 4091.28205128 4230.25641026 4488.20512821 4370.25641026
  4402.56410256 4461.02564103 4077.43589744 4378.46153846 4328.71794872
  3986.15384615 4461.02564103 4328.20512821 4203.58974359]
 [4393.84615385 4101.02564103 4193.33333333 4418.97435897 4270.25641026
  4392.30769231 4411.28205128 3982.56410256 4336.41025641 4213.33333333
  3930.25641026 4442.56410256 4261.02564103 4100.        ]
 [4396.41025641 4108.71794872 4210.76923077 4436.41025641 4310.76923077
  4401.02564103 4426.66666667 3980.5128205

In [26]:
# ============================================================
# Prepare DREAMER EEG + V/A/D
# ============================================================

import numpy as np
import pandas as pd

electrodes = [
    "AF3", "F7", "F3", "FC5",
    "T7", "P7", "O1", "O2",
    "P8", "T8", "FC6", "F4",
    "F8", "AF4"
]

dataset = []

for s in range(dreamer.noOfSubjects):

    subject = dreamer.Data[s]

    for v in range(dreamer.noOfVideoSequences):

        eeg = np.asarray(
            subject.EEG.stimuli[v],
            dtype=np.float32
        )

        eeg_df = pd.DataFrame(
            eeg,
            columns=electrodes
        )

        # Emotion labels
        valence = float(subject.ScoreValence[v])
        arousal = float(subject.ScoreArousal[v])
        dominance = float(subject.ScoreDominance[v])

        # Save trial
        dataset.append({
            "subject": s + 1,
            "trial": v + 1,
            "eeg": eeg_df,
            "valence": valence,
            "arousal": arousal,
            "dominance": dominance
        })


print("Total trials:", len(dataset))

Total trials: 414


In [27]:
#show and check dataset
print("Number of trials:", len(dataset))

print("\nFirst trial:")
print("Subject:", dataset[0]["subject"])
print("Trial:", dataset[0]["trial"])

print("\nV/A/D:")
print("Valence:", dataset[0]["valence"])
print("Arousal:", dataset[0]["arousal"])
print("Dominance:", dataset[0]["dominance"])

print("\nEEG shape:")
print(dataset[0]["eeg"].shape)

display(dataset[0]["eeg"].head())

Number of trials: 414

First trial:
Subject: 1
Trial: 1

V/A/D:
Valence: 4.0
Arousal: 3.0
Dominance: 2.0

EEG shape:
(25472, 14)


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4388.205078,4102.563965,4219.487305,4465.128418,4370.769043,4399.487305,4443.077148,4023.076904,4365.128418,4310.256348,3953.846191,4454.358887,4326.153809,4165.128418
1,4375.897461,4093.846191,4252.820312,4522.563965,4435.897461,4411.794922,4488.717773,4108.717773,4399.487305,4384.615234,4007.692383,4466.666504,4372.820312,4247.179688
2,4378.461426,4091.281982,4230.256348,4488.205078,4370.256348,4402.563965,4461.025879,4077.435791,4378.461426,4328.717773,3986.153809,4461.025879,4328.205078,4203.589844
3,4393.846191,4101.025879,4193.333496,4418.974121,4270.256348,4392.307617,4411.282227,3982.564209,4336.410156,4213.333496,3930.256348,4442.563965,4261.025879,4100.000000
4,4396.410156,4108.717773,4210.769043,4436.410156,4310.769043,4401.025879,4426.666504,3980.512939,4349.743652,4238.461426,3945.128174,4446.666504,4289.743652,4115.384766


In [28]:
# Window settings
window = 2  #seconds
overlap = 0.5

fs = int(dreamer.EEG_SamplingRate)

window_size = int(window * fs)
step_size = int(window_size * (1 - overlap))


# create window
windows = []

for start in range(0, len(eeg) - window_size + 1, step_size):
    
    end = start + window_size
    
    window = eeg[start:end]
    
    windows.append(window)

windows = np.array(windows)


print("Window size:", window_size)
print("Step size:", step_size)
print("Windows shape:", windows.shape)

Window size: 256
Step size: 128
Windows shape: (185, 256, 14)


In [29]:
from scipy.signal import butter, sosfiltfilt

# Sampling rate
fs = int(dreamer.EEG_SamplingRate)

# Filter settings
lowcut = 0.5
highcut = 45.0
filter_order = 4

# Band-pass filter
bpf = butter(
    filter_order,
    [lowcut, highcut],
    btype="bandpass",
    fs=fs,
    output="sos"
)

# filtering windows
filtered_windows = sosfiltfilt(
    bpf,
    windows,
    axis=1
)

filtered_windows = filtered_windows.astype(np.float32)

print("Original shape:", windows.shape)
print("Filtered shape:", filtered_windows.shape)
print("Data type:", filtered_windows.dtype)

Original shape: (185, 256, 14)
Filtered shape: (185, 256, 14)
Data type: float32


In [30]:
# number of windows
n_windows = filtered_windows.shape[0]

# calculate correlation for all windows
correlation_matrices = np.zeros(
    (n_windows, 14, 14),
    dtype=np.float32
)

for i in range(n_windows):
    correlation_matrices[i] = np.corrcoef(
        filtered_windows[i],
        rowvar=False
    )

print("Correlation matrices shape:", correlation_matrices.shape)

Correlation matrices shape: (185, 14, 14)


In [31]:
# test => window = filtered_windows[0]

# Sampling rate
fs = int(dreamer.EEG_SamplingRate)

print("Window shape:", window.shape)
print("Sampling rate:", fs)

Window shape: (256, 14)
Sampling rate: 128


In [32]:
# FFT in Window 
fft_values = np.fft.rfft(window, axis=0)


frequencies = np.fft.rfftfreq(
    window.shape[0],
    d=1 / fs
)

#check date
print("FFT shape:", fft_values.shape)
print("Frequency shape:", frequencies.shape)

print("\nFirst frequencies:")
print(frequencies[:15])

FFT shape: (129, 14)
Frequency shape: (129,)

First frequencies:
[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5 6.  6.5 7. ]


In [33]:
# calculate Power Spectrum (PSD) 
power_spectrum = np.abs(fft_values) ** 2

print("Power spectrum shape:", power_spectrum.shape)

print("\nAF3 power values:")
print(power_spectrum[:10, 0])

Power spectrum shape: (129, 14)

AF3 power values:
[1.2639145e+12 2.3500242e+05 5.9882690e+06 3.3688706e+05 4.5268366e+05
 7.0035331e+05 4.9369090e+04 3.4940090e+04 6.3516616e+03 1.1759696e+05]


In [34]:
from scipy.signal import welch

# Welch PSD for windows
freqs, psd = welch(
    window,
    fs=fs,
    axis=0,
    nperseg=256
)

print("Frequencies shape:", freqs.shape)
print("PSD shape:", psd.shape)

print("\nFirst frequencies:")
print(freqs[:15])

Frequencies shape: (129,)
PSD shape: (129, 14)

First frequencies:
[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5 6.  6.5 7. ]


In [35]:
#test

# define band of frequency
alpha_band = (8, 13)
beta_band = (13, 30)

# find frequency
alpha_idx = (freqs >= alpha_band[0]) & (freqs < alpha_band[1])
beta_idx = (freqs >= beta_band[0]) & (freqs <= beta_band[1])



print("Alpha frequencies:")
print(freqs[alpha_idx])

print("\nBeta frequencies:")
print(freqs[beta_idx])

Alpha frequencies:
[ 8.   8.5  9.   9.5 10.  10.5 11.  11.5 12.  12.5]

Beta frequencies:
[13.  13.5 14.  14.5 15.  15.5 16.  16.5 17.  17.5 18.  18.5 19.  19.5
 20.  20.5 21.  21.5 22.  22.5 23.  23.5 24.  24.5 25.  25.5 26.  26.5
 27.  27.5 28.  28.5 29.  29.5 30. ]


In [36]:
# number of windows
n_windows = filtered_windows.shape[0]

# output arrays
alpha_power_all = np.zeros(
    (n_windows, 14),
    dtype=np.float32
)

beta_power_all = np.zeros(
    (n_windows, 14),
    dtype=np.float32
)

# calculate power for each window
for i in range(n_windows):
    window = filtered_windows[i]

    # Welch PSD
    freqs, psd = welch(
        window,
        fs=fs,
        axis=0,
        nperseg=256
    )

    df = freqs[1] - freqs[0]

    # bands
    alpha_idx = (freqs >= 8) & (freqs < 13)
    beta_idx = (freqs >= 13) & (freqs <= 30)

    # Alpha Power
    alpha_power_all[i] = np.sum(
        psd[alpha_idx, :] * df,
        axis=0
    )

    # Beta Power
    beta_power_all[i] = np.sum(
        psd[beta_idx, :] * df,
        axis=0
    )


print("Alpha Power shape:", alpha_power_all.shape)
print("Beta Power shape:", beta_power_all.shape)

Alpha Power shape: (185, 14)
Beta Power shape: (185, 14)
